<a href="https://colab.research.google.com/github/bruno-antonio-marques/2026-2-gestao-e-analise-de-dados-exercicios/blob/main/Exerc%C3%ADcio_de_KNN_Exemplo_Supermercado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Parte 1

Criar Dataset, ou seja, tabela CSV com dados de cesta de mercado.

### Importar bibliotecas

In [12]:
import pandas as pd
import numpy as np

### Gerar dados aleatórios (semeaduras de dados) para o exercício

In [13]:
# Configurar semente para reprodutibilidade dos resultados na aula
np.random.seed(42)

### Criar função (método) **gera_cliente_por_perfil()**

In [14]:
def gerar_clientes_por_perfil(perfil, n_amostras):
    """
    Gera dados sintéticos baseados nas características de cada perfil de cliente.

    Atributos:
    - qtd_frescos: % de itens hortifrúti/frescos na cesta (0 a 100)
    - qtd_industrializados: % de itens ultraprocessados/congelados (0 a 100)
    - preco_medio_item: Valor médio gasto por item em R$
    - volume_total_itens: Quantidade total de itens no carrinho
    - %_itens_promocao: Porcentagem de itens comprados com desconto (0 a 100)
    """
    if perfil == "Cliente Saudável":
        frescos = np.random.normal(loc=70, scale=8, size=n_amostras)
        industrializados = np.random.normal(loc=10, scale=5, size=n_amostras)
        preco_medio = np.random.normal(loc=18, scale=3, size=n_amostras)
        volume = np.random.normal(loc=25, scale=5, size=n_amostras)
        promocao = np.random.normal(loc=15, scale=5, size=n_amostras)

    elif perfil == "Família Grande":
        frescos = np.random.normal(loc=30, scale=8, size=n_amostras)
        industrializados = np.random.normal(loc=45, scale=8, size=n_amostras)
        preco_medio = np.random.normal(loc=12, scale=2, size=n_amostras)
        volume = np.random.normal(loc=85, scale=12, size=n_amostras)
        promocao = np.random.normal(loc=40, scale=10, size=n_amostras)

    elif perfil == "Casal Jovem Gourmet":
        frescos = np.random.normal(loc=45, scale=7, size=n_amostras)
        industrializados = np.random.normal(loc=15, scale=5, size=n_amostras)
        preco_medio = np.random.normal(loc=35, scale=5, size=n_amostras)
        volume = np.random.normal(loc=20, scale=4, size=n_amostras)
        promocao = np.random.normal(loc=10, scale=4, size=n_amostras)

    elif perfil == "Solteiro Prático":
        frescos = np.random.normal(loc=10, scale=4, size=n_amostras)
        industrializados = np.random.normal(loc=70, scale=8, size=n_amostras)
        preco_medio = np.random.normal(loc=15, scale=3, size=n_amostras)
        volume = np.random.normal(loc=12, scale=3, size=n_amostras)
        promocao = np.random.normal(loc=20, scale=5, size=n_amostras)

    elif perfil == "Caçador de Ofertas":
        frescos = np.random.normal(loc=25, scale=6, size=n_amostras)
        industrializados = np.random.normal(loc=35, scale=8, size=n_amostras)
        preco_medio = np.random.normal(loc=8, scale=2, size=n_amostras)
        volume = np.random.normal(loc=35, scale=8, size=n_amostras)
        promocao = np.random.normal(loc=80, scale=8, size=n_amostras)

    df_perfil = pd.DataFrame({
        'pct_frescos': np.clip(frescos, 0, 100),
        'pct_industrializados': np.clip(industrializados, 0, 100),
        'preco_medio_item': np.clip(preco_medio, 1, None),
        'volume_total_itens': np.clip(volume, 1, None).astype(int),
        'pct_promocao': np.clip(promocao, 0, 100),
        'perfil_cliente': perfil
    })

    return df_perfil

### gerar amostra balanciada dentro dos 5 perfis em uma população de 100 individuos (clientes). Cada um dos 5 estratos possui 20 individuos.
Portanto as amostras serão balanceadas devido aos estratos terem todos a mesma proporção.

In [15]:
# Gerar 20 amostras para cada um dos 5 perfis (Total: 100 clientes)
perfis = ["Cliente Saudável", "Família Grande", "Casal Jovem Gourmet", "Solteiro Prático", "Caçador de Ofertas"]
dfs = [gerar_clientes_por_perfil(p, n_amostras=20) for p in perfis]

Combinar todos os estaratos em uma tabela em memória (DataFrame)

In [16]:
# Combinar em um único DataFrame
df_clientes = pd.concat(dfs, ignore_index=True)

### Arredondar valores numéricos

In [17]:
# Arredondar valores numéricos para melhor apresentação
df_clientes = df_clientes.round(2)

### Exibir os 5 primeiros (head ou cabeça da tabela) e salvar o outro conteúdo em formato csv.

In [18]:
# Exibir os 5 primeiros registros e salvar em CSV
print(df_clientes.head())
df_clientes.to_csv("clientes_supermercado_knn.csv", index=False)

   pct_frescos  pct_industrializados  preco_medio_item  volume_total_itens  \
0        73.97                 17.33             20.22                  22   
1        68.89                  8.87             18.51                  24   
2        75.18                 10.34             17.65                  19   
3        82.18                  2.88             17.10                  19   
4        68.13                  7.28             13.56                  29   

   pct_promocao    perfil_cliente  
0         13.90  Cliente Saudável  
1         16.79  Cliente Saudável  
2         22.39  Cliente Saudável  
3         12.41  Cliente Saudável  
4         10.96  Cliente Saudável  


### Listar tabela CSV criada na máquina virtual Linux do Colab.

In [19]:
! ls

clientes_supermercado_knn.csv  sample_data


### Listar qual é o meu diretório atual.

In [20]:
! pwd

/content


### Parte 2

Classificador KNN

### Importar bibliotecas

In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

### cARREGAR TABELA BALANCEADA DE 100 CLIENTES E 5 ESTRATOS NA MEMÓRIA

In [22]:
# 1. Carregar o dataset salvo anteriormente (ou definir a variável df_clientes)
df_clientes = pd.read_csv("clientes_supermercado_knn.csv")

### Separar atributos de entrada (x) e saída (y ou rotulo)

In [23]:
# 2. Separar Atributos (X) e Rótulo (y)
X = df_clientes.drop(columns=['perfil_cliente'])
y = df_clientes['perfil_cliente']

### Colocando os mesmos de escala do eixo x (entrada), e eixo y (saída) para todos os clientes(todas as linhas da tabela).

In [24]:
# 3. Normalização/Padronização dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Treinar modelo (classificar)

In [25]:
# 4. Treinar o KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_scaled, y)

KNeighborsClassifier()

### Testar (Submeter novo cliente nova linha) ao modelo de notificação

In [26]:
# 5. Testar uma nova predição
# Exemplo: %_frescos, %_industrializados, preco_medio, volume, %_promocao
novo_cliente = np.array([[12, 68, 14.50, 10, 15]])
novo_cliente_scaled = scaler.transform(novo_cliente)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [27]:
predicao = knn.predict(novo_cliente_scaled)
print(f"Perfil previsto para o novo cliente: {predicao[0]}")

Perfil previsto para o novo cliente: Solteiro Prático
